In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%%RecordEventWithColumnInfo
# %load_ext cudf.pandas
import sys, os
import pandas as pd
# point this at the directory *containing* tpch/
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))  
sys.path.insert(0, tpch_parent)

#from tpch import load_part, load_partsupp, load_supplier, load_nation,load_region,  q02
DATA_ROOT = "/home/dias-benchmarks/tpch/data"
STORAGE_OPTS = {}  # or load from JSON

In [ ]:
%%RecordEventWithColumnInfo
### cell 0 ###

# load just the datasets q01 needs:
def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%%RecordEventWithColumnInfo
### cell 1 ###

def load_partsupp(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/partsupp.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")  
    return df
partsupp = load_partsupp(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%%RecordEventWithColumnInfo
### cell 2 ###

def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%%RecordEventWithColumnInfo
### cell 3 ###

def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%%RecordEventWithColumnInfo
### cell 4 ###

def load_region(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/region.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
region = load_region(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%RecordEventWithColumnInfo
### cell 5 ###

nation_filtered = nation.loc[:, ["N_NATIONKEY", "N_NAME", "N_REGIONKEY"]]
region_filtered = region[(region["R_NAME"] == "EUROPE")]
region_filtered = region_filtered.loc[:, ["R_REGIONKEY"]]
r_n_merged = nation_filtered.merge(
    region_filtered, left_on="N_REGIONKEY", right_on="R_REGIONKEY", how="inner"
)
r_n_merged = r_n_merged.loc[:, ["N_NATIONKEY", "N_NAME"]]
supplier_filtered = supplier.loc[
    :,
    [
        "S_SUPPKEY",
        "S_NAME",
        "S_ADDRESS",
        "S_NATIONKEY",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
    ],
]
s_r_n_merged = r_n_merged.merge(
    supplier_filtered, left_on="N_NATIONKEY", right_on="S_NATIONKEY", how="inner"
)
s_r_n_merged = s_r_n_merged.loc[
    :,
    [
        "N_NAME",
        "S_SUPPKEY",
        "S_NAME",
        "S_ADDRESS",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
    ],
]
partsupp_filtered = partsupp.loc[:, ["PS_PARTKEY", "PS_SUPPKEY", "PS_SUPPLYCOST"]]
ps_s_r_n_merged = s_r_n_merged.merge(
    partsupp_filtered, left_on="S_SUPPKEY", right_on="PS_SUPPKEY", how="inner"
)
ps_s_r_n_merged = ps_s_r_n_merged.loc[
    :,
    [
        "N_NAME",
        "S_NAME",
        "S_ADDRESS",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
        "PS_PARTKEY",
        "PS_SUPPLYCOST",
    ],
]
part_filtered = part.loc[:, ["P_PARTKEY", "P_MFGR", "P_SIZE", "P_TYPE"]]
part_filtered = part_filtered[
    (part_filtered["P_SIZE"] == 15)
    & (part_filtered["P_TYPE"].str.endswith("BRASS"))
]
part_filtered = part_filtered.loc[:, ["P_PARTKEY", "P_MFGR"]]
merged_df = part_filtered.merge(
    ps_s_r_n_merged, left_on="P_PARTKEY", right_on="PS_PARTKEY", how="inner"
)
merged_df = merged_df.loc[
    :,
    [
        "N_NAME",
        "S_NAME",
        "S_ADDRESS",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
        "PS_SUPPLYCOST",
        "P_PARTKEY",
        "P_MFGR",
    ],
]
min_values = merged_df.groupby("P_PARTKEY", as_index=False, sort=False)[
    "PS_SUPPLYCOST"
].min()
min_values.columns = ["P_PARTKEY_CPY", "MIN_SUPPLYCOST"]
merged_df = merged_df.merge(
    min_values,
    left_on=["P_PARTKEY", "PS_SUPPLYCOST"],
    right_on=["P_PARTKEY_CPY", "MIN_SUPPLYCOST"],
    how="inner",
)
total = merged_df.loc[
    :,
    [
        "S_ACCTBAL",
        "S_NAME",
        "N_NAME",
        "P_PARTKEY",
        "P_MFGR",
        "S_ADDRESS",
        "S_PHONE",
        "S_COMMENT",
    ],
]
total = total.sort_values(
    by=["S_ACCTBAL", "N_NAME", "S_NAME", "P_PARTKEY"],
    ascending=[False, True, True, True],
)

In [ ]:
%%RecordEventWithColumnInfo
### cell 6 ###

total.info()